# PEYNNEY'S SOLUTION WITH AN EXTERNAL ELECTRIC FIELD IN EINSTEIN-MAXWELL-DILATON THEORIES

We consider external vacuum solutions in scalar–tensor
theories for the general class of actions

\begin{equation}
S = \int d^4x \sqrt{-g} \left[ R - 2 (\nabla \varphi)^2 - e^{-2\alpha\varphi}F^{2} \right],
\end{equation}

With the corresponding motion equations:

\begin{align*}
\nabla_\mu \left( e^{-2\alpha\phi} F^{\mu\nu} \right) = 0, \\
\nabla^2 \phi + \frac{a}{2} e^{-2\alpha\phi} F^2 = 0, \\
R_{\mu\nu} = 2 \nabla_\mu \phi \nabla_\nu \phi + 2 e^{-2\alpha\phi} F_{\mu\rho} F_\nu^{\ \rho} - \frac{1}{2} g_{\mu\nu} e^{-2\alpha\phi} F^2.
\end{align*}

Peynney (1968) provides the following static axisymmetric solution: 

\begin{align}
ds^{2} &= -\left(1-\frac{2m}{r}\right)dt^{2}
+ \exp\!\left(
-\frac{\xi^{2}m^{2}\,(r^{2}-2mr)\,\sin^{2}\theta}
{2\left(r^{2}-2mr+m^{2}\cos^{2}\theta\right)^{2}}
\right)
\left[
\left(1-\frac{2m}{r}\right)^{-1}dr^{2}
+ r^{2}d\theta^{2}
\right]
+ r^{2}\sin^{2}\theta\, d\varphi^{2}.
\\[1ex]
\varphi &= \frac{m\xi}{\sqrt{r^{2}-2mr+m^{2}\cos^{2}\theta}}.
\end{align}

/!\ Note that within the notebook the constant $\xi$ is written $c$.

In [1]:
%display latex

In [2]:
version()

'SageMath version 10.1, Release Date: 2023-08-20'

'SageMath version 10.1, Release Date: 2023-08-20'

In [3]:
from sage.manifolds.operators import dalembertian
from sage.manifolds.operators import laplacian
from sage.manifolds.operators import grad
from sage.symbolic.operators import FDerivativeOperator
import sage_func

In [4]:
functions_list=['U','V','X','Y','Lambda','Av','Ev']
def subs_func(arg, full=True):
    subs_funcs = [(U, A_exp), (V, B_exp), (X, C_exp), (Y, D_exp),(Lambda,Lambda_expr),(Av,Av_expr),(Ev,Ev_expr)] \
                                        if full else [(U, A_exp), (V, B_exp), (X, C_exp), (Y, D_exp)]
    if hasattr(arg, 'expr'):
        arg = arg.expr()
    if hasattr(arg, 'apply_map')*hasattr(arg, 'display'):
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg.apply_map(lambda f: f.substitute_function(old_func, new_func).factor())
            print('{0} substituted'.format(functions_list[i])) if full else None
    elif hasattr(arg, 'apply_map'):
        return arg.apply_map(lambda f: subs_func(f).factor())
    else:
        for i, (old_func, new_func) in enumerate(subs_funcs):
            arg = arg.substitute_function(old_func, new_func).factor()
            print('{0} substituted'.format(functions_list[i])) if full else None
    print('Full Substitution : Done') if full else print('Partial Substitution : Done')
    return arg

In [5]:
M = Manifold(4, 'M', structure='Lorentzian')
print(M)

4-dimensional Lorentzian manifold M


In [6]:
XY.<t,r,th,ph> = M.chart(r"t r:(0,+oo) th:(0,pi):\theta ph:(0,2*pi):\varphi")
XY

Chart (M, (t, r, th, ph))

# I. Definition of the metric

In [7]:
U = function('U')
V = function('V')
X = function('X')
Y = function('Y')

In [8]:
g = M.metric()

g[0,0] = U(r,th)
g[1,1] = V(r,th)
g[2,2] = X(r,th)
g[3,3] = Y(r,th)

show(g.display())

g = U(r, th) dt⊗dt + V(r, th) dr⊗dr + X(r, th) dth⊗dth + Y(r, th) dph⊗dph

In [34]:
nab = g.connection(name=r'\nabla') 

# II. The vector potential

In [35]:
m, c, a, B = var('m c alpha B')
assume(m >= 0, 'real')
assume(c > 0, 'real')

pot_vec = M.tensor_field(0,1,name='A')
pot_vec[0]=((a*(2)^(1/2)/2)*(m^2*cos(th)^2-2*m*r+r^2)^(-1/2)*(B*m^2-B*m*r)*cos(th)*c+B*r*cos(th)-2*B*m*cos(th)).factor()
pot_vec[1]=0
pot_vec[2]=0
pot_vec[3]=0
pot_vec.apply_map(lambda f:f.simplify_full())

show(pot_vec.display())

A = -1/2*sqrt(2)*(sqrt(m^2*cos(th)^2 - 2*m*r + r^2)*(2*sqrt(2)*B*m - sqrt(2)*B*r)*cos(th) - (B*alpha*c*m^2 - B*alpha*c*m*r)*cos(th))/sqrt(m^2*cos(th)^2 - 2*m*r + r^2) dt

In [36]:
DF = nab(pot_vec) ; DF

Tensor field \nabla(A) of type (0,2) on the 4-dimensional Lorentzian manifold M

# III. Definition of the EM tensor

Here, it was necessary to manually replace $cos^{3}(\theta)$ by $cos(\theta)(1 - sin^{2}(\theta))$ otherwise, SageMath is unable to simplify the expression on its own, and the growing complexity of the field equations prevents it from verifying the solution.

In [37]:
F = diff(pot_vec)
F.apply_map(lambda f:f.factor().simplify_trig().subs({sqrt(2)*B*a*c*m**3*cos(th)**Integer(3): 
            sqrt(2)*B*a*c*m**3*cos(th) - sqrt(2)*B*a*c*m**3*cos(th)*sin(th)**Integer(2)}))
F.set_name('F')
Fuu = F.up(g)

F.display()

F = -1/2*(sqrt(2)*B*alpha*c*m^3*cos(th)*sin(th)^2 + 2*(B*m^2*cos(th)^3 - (2*B*m*r - B*r^2)*cos(th))*sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/(m^2*cos(th)^2 - 2*m*r + r^2)^(3/2) dt∧dr + 1/2*(2*(4*B*m^2*r - 4*B*m*r^2 + B*r^3 - (2*B*m^3 - B*m^2*r)*cos(th)^2)*sqrt(m^2*cos(th)^2 - 2*m*r + r^2)*sin(th) - (2*sqrt(2)*B*alpha*c*m^3*r - 3*sqrt(2)*B*alpha*c*m^2*r^2 + sqrt(2)*B*alpha*c*m*r^3)*sin(th))/(m^2*cos(th)^2 - 2*m*r + r^2)^(3/2) dt∧dth

In [38]:
E_elec = M.tensor_field(1,0)
for i in [1, 2, 3]:  
    E_elec[i] = F[0, i].expr().canonicalize_radical() 
show(LatexExpr(r'E^{\mu} = '),E_elec[:])

E^{\mu} =  [0,
 -1/2*(sqrt(2)*B*alpha*c*m^3*cos(th)*sin(th)^2 + 2*(B*m^2*cos(th)^3 - (2*B*m*r - B*r^2)*cos(th))*sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/(m^2*cos(th)^2 - 2*m*r + r^2)^(3/2),
 1/2*(2*(4*B*m^2*r - 4*B*m*r^2 + B*r^3 - (2*B*m^3 - B*m^2*r)*cos(th)^2)*sqrt(m^2*cos(th)^2 - 2*m*r + r^2)*sin(th) - (2*sqrt(2)*B*alpha*c*m^3*r - 3*sqrt(2)*B*alpha*c*m^2*r^2 + sqrt(2)*B*alpha*c*m*r^3)*sin(th))/(m^2*cos(th)^2 - 2*m*r + r^2)^(3/2),
 0]

# IV. Variables and functions

In [39]:
Av = function('A_v')
Ev = function('E_v')
Lambda = function('Lambda')

In [40]:
Av_expr(r) = (1- 2*m /r)
Ev_expr(r,th) = exp(- (c**2 * m**2 * r**2 * Av_expr(r) * sin(th)**2) / (2 * (r**2*Av_expr(r) + m**2 * cos(th)**2)**2))
old_Phi = M.scalar_field({XY:m*c / sqrt(2*(r**2*Av_expr(r) + m**2 * cos(th)**2))}, name=r'\varphi')
old_Psi = exp(2*alpha*old_Phi)
Lambda_expr(r, th) = 1 + (1+ alpha**2) / 4 * B**2 * (r*sin(th))**2 * old_Psi.expr()

In [41]:
show(LatexExpr(r'\Lambda = '), Lambda_expr(r, th))
show(LatexExpr(r'A_v(r) = '), Av_expr(r), LatexExpr(r', \quad\quad    E_v(r,\theta) = '), Ev_expr(r,th))
show(LatexExpr(r"\phi_{0} = "), old_Phi.expr())
show(LatexExpr(r"\Psi_{0} = e^{2\alpha\phi_{0}} = "), old_Psi.expr())

\Lambda =  1/4*(alpha^2 + 1)*B^2*r^2*e^(sqrt(2)*alpha*c*m/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))*sin(th)^2 + 1

A_v(r) =  -2*m/r + 1 , \quad\quad    E_v(r,\theta) =  e^(1/2*c^2*m^2*r^2*(2*m/r - 1)*sin(th)^2/(m^2*cos(th)^2 - r^2*(2*m/r - 1))^2)

\phi_{0} =  c*m/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1))

\Psi_{0} = e^{2\alpha\phi_{0}} =  e^(sqrt(2)*alpha*c*m/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

# V. Metric and potential vector expressions

In [42]:
A_exp(r,th) = - Av(r) * Lambda(r,th)**(2/(1+alpha**2))
B_exp(r,th) = Ev(r,th) / Av(r) * Lambda(r,th)**(2/(1+alpha**2))
C_exp(r,th) = r**2 * Ev(r,th) * Lambda(r,th)**(2/(1+alpha**2))
D_exp(r,th) =  (r*sin(th))**2 / Lambda(r,th)**(2/(1+alpha**2))

show(LatexExpr(r"A_{\mu\nu} = 0 \rightarrow A'_{\mu\nu} ="), pot_vec.display())
show(LatexExpr(r'\sqrt{-g} = '),subs_func((-g.det().expr())^(1/2), full=False).canonicalize_radical())

A_{\mu\nu} = 0 \rightarrow A'_{\mu\nu} = A = -1/2*sqrt(2)*(sqrt(m^2*cos(th)^2 - 2*m*r + r^2)*(2*sqrt(2)*B*m - sqrt(2)*B*r)*cos(th) - (B*alpha*c*m^2 - B*alpha*c*m*r)*cos(th))/sqrt(m^2*cos(th)^2 - 2*m*r + r^2) dt

Partial Substitution : Done


\sqrt{-g} =  r^2*Lambda(r, th)^(2/(alpha^2 + 1))*E_v(r, th)*abs(sin(th))

In [43]:
g_nod = g.copy(name='g')
g_nod = subs_func(g_nod, full=False)
show(g_nod.display())
g_nod = subs_func(g_nod)
nod_ratio = (g_nod[3,3]/(sin(th)**2*g_nod[2,2])).factor()
show(LatexExpr(r"\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = " + latex(limit(nod_ratio.expr(), th=0).canonicalize_radical())))

Partial Substitution : Done


g = -Lambda(r, th)^(2/(alpha^2 + 1))*A_v(r) dt⊗dt + Lambda(r, th)^(2/(alpha^2 + 1))*E_v(r, th)/A_v(r) dr⊗dr + r^2*Lambda(r, th)^(2/(alpha^2 + 1))*E_v(r, th) dth⊗dth + r^2*sin(th)^2/Lambda(r, th)^(2/(alpha^2 + 1)) dph⊗dph

U substituted
V substituted
X substituted
Y substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


\lim_{\theta\rightarrow 0} \frac{g_{\phi\phi}}{\sin^{2}\theta g_{\theta\theta}} = 1

In [44]:
Phi = M.scalar_field({XY:-alpha/ (1+alpha**2) * log(Lambda(r,th))+old_Phi.expr()}, name=r'\phi') 
Psi = M.scalar_field({XY:old_Psi.expr() / (Lambda(r,th))**(2*alpha**2/(1+alpha**2))}, name=r'\Psi')

show(LatexExpr(r'\varphi = '), old_Phi.expr(), LatexExpr(r"\rightarrow \varphi' = "), Phi.expr())
show(LatexExpr(r'\Psi = '), old_Psi.expr(), LatexExpr(r"\rightarrow \Psi' = "), Psi.expr())

\varphi =  c*m/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) \rightarrow \varphi' =  c*m/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) - alpha*log(Lambda(r, th))/(alpha^2 + 1)

\Psi =  e^(sqrt(2)*alpha*c*m/sqrt(m^2*cos(th)^2 - 2*m*r + r^2)) \rightarrow \Psi' =  e^(sqrt(2)*alpha*c*m/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/Lambda(r, th)^(2*alpha^2/(alpha^2 + 1))

In [45]:
Phi_elec = M.scalar_field({XY: -Phi.expr()}, name=r'\phi') 
Psi_elec = M.scalar_field({XY: 1 / (old_Psi.expr())*(Lambda(r,th))**(2*alpha**2/(1+alpha**2))}, name=r'\Psi')

show(LatexExpr(r'\varphi = '), Phi.expr(), LatexExpr(r"\rightarrow \varphi^{e} = "), Phi_elec.expr())
show(LatexExpr(r'\Psi = '), Psi.expr(), LatexExpr(r"\rightarrow \Psi^{e} = "), Psi_elec.expr())

\varphi =  c*m/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) - alpha*log(Lambda(r, th))/(alpha^2 + 1) \rightarrow \varphi^{e} =  -c*m/sqrt(2*m^2*cos(th)^2 - 2*r^2*(2*m/r - 1)) + alpha*log(Lambda(r, th))/(alpha^2 + 1)

\Psi =  e^(sqrt(2)*alpha*c*m/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))/Lambda(r, th)^(2*alpha^2/(alpha^2 + 1)) \rightarrow \Psi^{e} =  Lambda(r, th)^(2*alpha^2/(alpha^2 + 1))*e^(-sqrt(2)*alpha*c*m/sqrt(m^2*cos(th)^2 - 2*m*r + r^2))

# VI. The field equations

## VI.1. Verification of the Maxwell equations $\bigtriangledown_{\mu}(e^{-2\alpha\Phi}F^{\mu\nu})=0$

In [46]:
eq1 = nab(Fuu / Psi_elec)
eq = eq1['^a._a']

In [47]:
eq.apply_map(lambda f:f.factor())
eq = subs_func(eq)
show('Substitution: done')

U substituted
V substituted
X substituted
Y substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


'Substitution: done'

In [48]:
show('Verification: ', LatexExpr(r'\bigtriangledown_{\mu}(e^{-2\alpha\Phi}F^{\mu\nu})=0: '), eq == 0)

'Verification: ' \bigtriangledown_{\mu}(e^{-2\alpha\Phi}F^{\mu\nu})=0:  True

## VI. 2. Verification of the equation $\bigtriangledown^{2}\Phi+\frac{\alpha}{2}e^{-2\alpha\Phi}F^{2}=0$ 

In [49]:
g = subs_func(g,full=False) 

Partial Substitution : Done


In [50]:
F2 = F['_ab']*Fuu['^ab']
eq2_1 = Phi_elec.dalembertian()
eq2_2 = (a/2)*F2 / Psi_elec 
eq2 = eq2_1 + eq2_2

In [51]:
eq_num = eq2.expr().factor()
eq_num = numerator(subs_func(eq_num))

U substituted
V substituted
X substituted
Y substituted
Lambda substituted
Av substituted
Ev substituted
Full Substitution : Done


In [52]:
LateX = LatexExpr(r'\bigtriangledown^{2}\Phi+\frac{\alpha}{2}e^{-2\alpha\Phi}F^{2}=0 ')
show('Verification: ', LateX, LatexExpr(r'\quad'), eq_num.is_zero())

'Verification: ' \bigtriangledown^{2}\Phi+\frac{\alpha}{2}e^{-2\alpha\Phi}F^{2}=0  \quad True

## VI.3 Verification of the main motion equation $R_{\mu\nu}=2\triangledown_{\mu}\Phi\triangledown_{\nu}\Phi+\frac{T_{\mu\nu}}{\Psi}$

In [53]:
Fud = F.up(g,0)
T = 2*(F['_k.']*Fud['^k_.'] - 1/4*F2*g)
UU=T / Psi_elec 

In [54]:
nab_phi = nab(Phi_elec)
S = 2*nab_phi*nab_phi
RHS = UU + S

In [55]:
ER_ricc = g.ricci()

In [56]:
RHS = subs_func(RHS, full=False)

Partial Substitution : Done


In [57]:
eq3 = ER_ricc - RHS

In [58]:
eq3.apply_map(lambda f :f.substitute_function(Lambda,Lambda_expr).substitute_function(Av,Av_expr)\
                  .substitute_function(Ev,Ev_expr).factor())

In [59]:
LateX = LatexExpr(r'R_{\mu\nu}=2\triangledown_{\mu}\Phi\triangledown_{\nu}\Phi+\frac{T_{\mu\nu}}{\Psi} ')
show('Verification: ',LateX, LatexExpr(r'\quad'), eq3.is_zero())

'Verification: ' R_{\mu\nu}=2\triangledown_{\mu}\Phi\triangledown_{\nu}\Phi+\frac{T_{\mu\nu}}{\Psi}  \quad True

# WE CONCLUDE THAT EMBEDDING THE PENNEY SOLUTION INTO AN ELECTRIC FIELD IS SOLUTION IN EINSTEIN-MAXWELL-DILATON THEORIES FOR ARBITRARY COUPLING